# Auto Research Agent — Simplified Multi-Agent Research & Report System

Free-tier only. No card required anywhere.

**Stack:** Groq (LLM, free) · DuckDuckGo Search (free, no key) · Chroma (local RAG) ·
HuggingFace embeddings (local) · LangGraph (orchestration) · Gradio (UI)

**Flow:** `Planner -> Researcher -> Analyst -> Writer`

Get your free Groq key at https://console.groq.com/keys (no card needed).


In [ ]:
!pip install -q langgraph langchain langchain-groq langchain-community \
    langchain-text-splitters langchain-huggingface langchain-chroma \
    ddgs chromadb sentence-transformers gradio pypdf

In [ ]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Enter Groq API key: ")


## 1. LLM + Embeddings

In [ ]:
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.2)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


## 2. RAG store (optional docs)

Drop PDFs in `/content/docs` (or mount Drive) before running this cell. Skip if you don't need document grounding for the task.

In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

DOCS_DIR = "/content/docs"
PERSIST_DIR = "/content/chroma_db"

def build_vectorstore():
    if not os.path.isdir(DOCS_DIR) or not os.listdir(DOCS_DIR):
        print("No docs found — skipping RAG store, web search only.")
        return None
    docs = []
    for f in os.listdir(DOCS_DIR):
        if f.lower().endswith(".pdf"):
            docs.extend(PyPDFLoader(os.path.join(DOCS_DIR, f)).load())
    splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
    chunks = splitter.split_documents(docs)
    vs = Chroma.from_documents(chunks, embeddings, persist_directory=PERSIST_DIR)
    print(f"Indexed {len(chunks)} chunks from {len(docs)} pages.")
    return vs

vectorstore = build_vectorstore()
retriever = vectorstore.as_retriever(search_kwargs={"k": 4}) if vectorstore else None


## 3. Tools — web search + code execution

In [ ]:
from ddgs import DDGS  # duckduckgo_search was renamed to ddgs upstream

def web_search(query: str, max_results: int = 5) -> str:
    # DDG is free/no-key but scraping-based, so it rate-limits under bursts.
    # Don't let that crash the whole agent run.
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=max_results))
    except Exception as e:
        return f"Web search unavailable right now ({e.__class__.__name__}). Relying on indexed documents only."
    if not results:
        return "No results found."
    return "\n\n".join(f"- {r['title']}: {r['body']} ({r['href']})" for r in results)


In [ ]:
import io
import contextlib

# Demo-grade sandbox: restricted builtins, no file/network access.
# Not production-safe — fine for a personal portfolio project, don't expose publicly as-is.
SAFE_BUILTINS = {
    "print": print, "range": range, "len": len, "sum": sum, "min": min, "max": max,
    "abs": abs, "round": round, "sorted": sorted, "enumerate": enumerate, "zip": zip,
    "list": list, "dict": dict, "set": set, "tuple": tuple, "str": str, "int": int,
    "float": float, "bool": bool,
}

def run_python(code_str: str) -> str:
    buf = io.StringIO()
    local_env = {}
    try:
        with contextlib.redirect_stdout(buf):
            exec(code_str, {"__builtins__": SAFE_BUILTINS, "pd": __import__("pandas"),
                             "np": __import__("numpy")}, local_env)
    except Exception as e:
        return f"Error: {e}"
    output = buf.getvalue()
    return output if output else "Code ran with no printed output."


## 4. Agent state + nodes (LangGraph)

In [ ]:
from typing import TypedDict, Optional
from langgraph.graph import StateGraph, END

class AgentState(TypedDict):
    task: str
    plan: str
    research: str
    analysis: str
    report: str


In [ ]:
def planner_node(state: AgentState) -> AgentState:
    prompt = f"""Break this task into a short plan: what to research, and whether
any data analysis/code execution is needed. Task: {state['task']}
Reply in 3-5 bullet points, plain text."""
    plan = llm.invoke(prompt).content
    return {**state, "plan": plan}


def researcher_node(state: AgentState) -> AgentState:
    web_results = web_search(state["task"])
    doc_results = ""
    if retriever:
        docs = retriever.invoke(state["task"])
        doc_results = "\n\n".join(d.page_content for d in docs)
    combined = f"WEB RESULTS:\n{web_results}\n\nDOCUMENT RESULTS:\n{doc_results or 'N/A'}"
    return {**state, "research": combined}


def analyst_node(state: AgentState) -> AgentState:
    prompt = f"""Given this research, decide if a short Python snippet would help
(e.g. quick math/aggregation). If yes, write ONLY the code, nothing else.
If no analysis is needed, reply exactly: NO_CODE_NEEDED

Research:
{state['research'][:3000]}"""
    code_reply = llm.invoke(prompt).content.strip()
    if code_reply == "NO_CODE_NEEDED" or "NO_CODE_NEEDED" in code_reply:
        return {**state, "analysis": "No code analysis needed."}
    # strip markdown fences if present
    cleaned = code_reply.replace("```python", "").replace("```", "").strip()
    result = run_python(cleaned)
    return {**state, "analysis": f"Code:\n{cleaned}\n\nOutput:\n{result}"}


def writer_node(state: AgentState) -> AgentState:
    prompt = f"""Write a concise Markdown report answering the task below.
Use the research and analysis provided. Keep it well-structured with headers.

Task: {state['task']}

Plan: {state['plan']}

Research: {state['research'][:4000]}

Analysis: {state['analysis']}"""
    report = llm.invoke(prompt).content
    return {**state, "report": report}


## 5. Build the graph (linear: planner -> researcher -> analyst -> writer)

In [ ]:
graph = StateGraph(AgentState)
graph.add_node("planner", planner_node)
graph.add_node("researcher", researcher_node)
graph.add_node("analyst", analyst_node)
graph.add_node("writer", writer_node)

graph.set_entry_point("planner")
graph.add_edge("planner", "researcher")
graph.add_edge("researcher", "analyst")
graph.add_edge("analyst", "writer")
graph.add_edge("writer", END)

app = graph.compile()


## 6. Run it

In [ ]:
task = "Summarize the current state of open-source Arabic NLP models and list the top 3 by benchmark performance."
result = app.invoke({"task": task, "plan": "", "research": "", "analysis": "", "report": ""})
print(result["report"])


## 7. Gradio UI

In [ ]:
import gradio as gr

def run_agent(task_input):
    out = app.invoke({"task": task_input, "plan": "", "research": "", "analysis": "", "report": ""})
    return out["report"]

demo = gr.Interface(
    fn=run_agent,
    inputs=gr.Textbox(label="Task", lines=2, placeholder="e.g. Compare pricing of top 3 cloud GPU providers"),
    outputs=gr.Markdown(label="Report"),
    title="Auto Research Agent",
    description="Planner -> Researcher (web + RAG) -> Analyst (code) -> Writer",
)

demo.launch(share=True)


## Notes for the repo

- README should state the sandbox limitation honestly (restricted exec, not a real sandbox).
- Add a `docs/` folder with 1-2 sample PDFs so the RAG path is demoable out of the box.
- Record a 1-2 min Loom/screen recording of the Gradio demo, link it at top of README.
- CV line (honest, not inflated):
  *"Built a multi-agent research pipeline (LangGraph) combining web search, local RAG (Chroma),
  and code-executing analysis to auto-generate research reports; deployed via Gradio."*
